# NVFP4


Before understanding NVFP4, you first need to understand what actually limits LLM performance.

Most people assume the GPU is busy doing matrix multiplication.

For large language models, that's only partly true.

The real bottleneck is usually:

GPU Memory (HBM)
        ↓
Load weights
        ↓
Tensor Cores
        ↓
Compute
        ↓
Write results

For example:

Hidden size = 8192

Weight matrix:

8192 × 8192

≈67 million weights

If stored in FP16:

67M × 2 bytes
≈134 MB

Why not just use FP4?

The obvious idea is:

FP16 → FP4

2 bytes
↓

0.5 bytes

That gives:

134 MB

↓

33.5 MB

That's a 4× reduction in memory traffic.

Understanding FP4

A floating-point number has three parts:

Sign
Exponent
Mantissa

Conceptually:

value = (-1)^sign × mantissa × 2^exponent

With FP16, you have enough bits to represent a wide range of values.

With FP4, you only have 4 total bits.

A simplified view:

S E E M

1 sign
2 exponent
1 mantissa

Only 16 possible bit patterns exist.

That means only 16 representable values.

For example (illustrative, not exact NVFP4 encoding):

0
±0.5
±1
±2
±4
±8
...

Now consider a real neural-network weight:

0.127

FP4 cannot represent it exactly.

It may become

0.125

or

0.25

depending on the encoding.

That error is called quantization error.

Where NVFP4 fits in

NVFP4 is NVIDIA's hardware-native FP4 format designed specifically for Blackwell Tensor Cores. It combines:

A 4-bit floating-point representation for weights.
Efficient per-block scaling handled directly in hardware.
Tensor Core instructions that can read FP4 values, apply the block scale, and perform matrix multiplication in a single pipeline, accumulating results in higher precision (such as FP16 or FP32).

The crucial point is that the Tensor Cores understand the format natively. Earlier GPUs could emulate low-bit formats in software, but Blackwell performs the unpacking, scaling, multiplication, and accumulation as dedicated hardware operations, dramatically reducing overhead.